In [ ]:
import sys; sys.path.append('..')
import MeshFEM, mesh, utils, mesh_utilities, visualization, importlib
from py_newton_optimizer import NewtonOptimizerOptions
from io_redirection import suppress_stdout
import wall_width_formulas as wwf
import parametrization
import inflation, sheet_meshing, wall_generation
import numpy as np
from matplotlib import pyplot as plt
from tri_mesh_viewer import TriMeshViewer as Viewer

target_surf = mesh.Mesh('../../examples/lilium.msh')
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=True))
target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [ ]:
tsview = Viewer(target_surf)
tsview.showWireframe()
tsview.show()

In [ ]:
# Choose reasonable stretching bounds
alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(2, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
print(alphaMin, alphaMax)

In [ ]:
lg = parametrization.LocalGlobalParametrizer(target_surf, parametrization.lscm(target_surf))

for i in range(1000): lg.runIteration()
print(lg.energy())
lg.alphaMin = 1.4
lg.alphaMax = np.pi / 2

print(lg.energy())
lg.runIteration()
print(lg.energy())

for i in range(50): lg.runIteration()
print(lg.energy())

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(target_surf, lg.uv())
rparam.alphaMin = alphaMin
rparam.alphaMax = alphaMax

# Select scale- and mesh-independent energy formulation
rparam.scaleInvariantFittingEnergy = True
rparam.dualLaplacianStencil.type = rparam.dualLaplacianStencil.Type.DualMeshIDT

In [ ]:
def optimize_rparam(param, alphaRegW, phiRegW, bendRegWeight):
    param.alphaRegW = alphaRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegWeight
    opts = NewtonOptimizerOptions()
    opts.useIdentityMetric = True
    opts.beta = 1e-4
    opts.niter = 2000
    opts.gradTol = 1e-14
    parametrization.benchmark_reset()
    cr = parametrization.regularized_parametrization_newton(param, param.rigidMotionPinVars, opts)
    parametrization.benchmark_report()

In [ ]:
importlib.reload(utils)
utils.normalizedParamEnergies(rparam)

In [ ]:
visualization.singularValueHistogram(rparam)

In [ ]:
parametrization.benchmark_reset()
#with suppress_stdout(): optimize_rparam(rparam, 1e-6, 1e-6, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-5, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-4, 1e-4, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-3, 1e-3, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-4, 1e-4, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-5, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-2, 1e-2, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-3, 1e-3, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-4, 1e-4, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-5, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-6, 1e-6, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-7, 1e-7, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-6, 1e-6, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-5, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-4, 1e-4, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-3, 1e-3, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-2, 1e-2, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-1, 1e-1, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-2, 1e-2, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-3, 1e-3, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-4, 1e-4, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-5, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-6, 1e-6, 0)
#with suppress_stdout(): optimize_rparam(rparam, 1e-7, 1e-7, 0)
with suppress_stdout(): optimize_rparam(rparam, 1e-2, 1e-2, 0)
with suppress_stdout(): optimize_rparam(rparam, 1e-3, 1e-3, 0)
with suppress_stdout(): optimize_rparam(rparam, 1e-4, 1e-4, 0)
with suppress_stdout(): optimize_rparam(rparam, 1e-5, 1e-5, 0)
parametrization.benchmark_report()

In [ ]:
visualization.visualize(rparam)

In [ ]:
importlib.reload(visualization);

In [ ]:
import point_cloud_utils as pcu

In [ ]:
visualization.visualizeChannelOrientationSubsampled(rparam, numSamples=1500, orientationHue=True)

In [ ]:
widths = wwf.wallWidthForCanonicalWidth(wwf.canonicalWallWidthForStretchFactor(rparam.getAlphas()), 10)
(np.min(widths), np.max(widths))

In [ ]:
np.min(wwf.canonicalWallWidthForStretchFactor(rparam.getAlphas())) * (10 / (2 * np.pi))

In [ ]:
np.max(wwf.canonicalWallWidthForStretchFactor(rparam.getAlphas())) * (10 / (2 * np.pi))

In [ ]:
wwf.canonicalWallWidthForStretchFactor(rparam.getAlphas())

In [ ]:
np.min(rparam.getAlphas())

In [ ]:
np.max(rparam.getAlphas())

## Upsampling and channel generation

In [ ]:
nsubdiv=3
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)
(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_stripe_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles,
                                                                    wwf.canonicalWallWidthForStretchFactor(upsampledStretches), frequency=0.5)

In [ ]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, height=12)

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=2.5,
                                              minContourLen=20)

In [ ]:
visualization.plot_line_segments(pts, edges, width=20, height=16)

In [ ]:
importlib.reload(sheet_meshing);
m, iwv, iwbv = sheet_meshing.generateSheetMesh(sdfVertices, sdfTris, sdf, triArea=2.5, targetEdgeSpacing=2.5, minContourLen=20)